In [1]:
import h5py
import numpy as np
import numpy.typing as npt
import random
import operator
from collections.abc import MutableSequence
import numba

import tensorflow as tf

2025-05-12 00:46:00.787438: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-12 00:46:00.944800: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-12 00:46:00.944829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-12 00:46:00.956505: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-12 00:46:00.986976: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-12 00:46:00.987453: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [2]:
with h5py.File("../docker/output/imdd/CM_imdd-M64-n48-N_tx25-N_rx15-17ps.hd5", "r") as f:
    confusion_matrix = np.array(f["confusion_matrix"], dtype=np.int32)

M = len(confusion_matrix)

In [36]:
@numba.njit
def hamming_distance(x: int, y: int) -> int:
    xor = x ^ y
    distance = 0

    while xor != 0:
        distance += xor & 1
        xor >>= 1

    return distance


@numba.njit
def make_hamming_matrix(M: int) -> npt.NDArray:
    hamming_matrix = np.zeros((M, M), dtype=np.int16)

    for i in range(M):
        for j in range(M):
            hamming_matrix[i, j] = hamming_distance(i, j)

    return hamming_matrix


@numba.njit
def compute_total_errors(
    mapping: npt.NDArray,
    confusion_matrix: npt.NDArray,
    hamming_matrix: npt.NDArray
) -> int:
    total_errors = 0
    M = len(mapping)

    for i in range(M):
        mapped_i = mapping[i]

        for j in range(M):
            mapped_j = mapping[j]
            total_errors += confusion_matrix[i, j] * hamming_matrix[mapped_i, mapped_j]

    return total_errors


@numba.njit
def random_swap(x: npt.NDArray, n: int = 1) -> None:
    for i in range(n):
        i = random.randint(0, len(x) - 1)

        while True:
            j = random.randint(0, len(x) - 1)

            if j != i:
                break

        x[i], x[j] = x[j], x[i]


@numba.njit
def optimize_greedy(max_iterations: int, confusion_matrix: npt.NDArray, initial_mapping: npt.NDArray) -> npt.NDArray:
    M = len(confusion_matrix)
    hamming_matrix = make_hamming_matrix(M)

    mapping = np.copy(initial_mapping)

    current_errors = compute_total_errors(mapping, confusion_matrix, hamming_matrix)
    best_candidate, best_metric = mapping, current_errors

    for i in range(max_iterations):
        stagnating = True
        
        for j in range(M):
            for k in range(j + 1, M):
                candidate = np.copy(mapping)
                candidate[i], candidate[j] = candidate[j], candidate[i]

                metric = compute_total_errors(candidate, confusion_matrix, hamming_matrix)
                if metric <= best_metric:
                    best_candidate, best_metric = candidate, metric
                    stagnating = False

        if stagnating:
            break

    return best_candidate


@numba.njit
def optimize_random_search(n_iterations: int, confusion_matrix: npt.NDArray, initial_mapping: npt.NDArray) -> npt.NDArray:
    M = len(confusion_matrix)
    hamming_matrix = make_hamming_matrix(M)
    
    best_mapping = np.copy(initial_mapping)
    best_metric = compute_total_errors(initial_mapping, confusion_matrix, hamming_matrix)
    
    for i in range(n_iterations):
        new_mapping = np.copy(initial_mapping)
        np.random.shuffle(new_mapping)
        new_metric = compute_total_errors(new_mapping, confusion_matrix, hamming_matrix)

        if new_metric < best_metric:
            best_mapping = new_mapping
            best_metric = new_metric

    return best_mapping


def optimize_genetic(n_iterations: int, confusion_matrix: npt.NDArray, initial_mapping: npt.NDArray) -> npt.NDArray:
    max_candidates = 10000
    max_survivors = 9000
    M = len(confusion_matrix)

    hamming_matrix = make_hamming_matrix(M)

    candidates = [(
        compute_total_errors(initial_mapping, confusion_matrix, hamming_matrix),
        np.copy(initial_mapping)
    )]

    key_function = operator.itemgetter(0)

    for i in range(n_iterations):
        if (i + 1) % 2000 == 0:
            print(f"Iteration {i + 1}, error_count: {candidates[0][0]}")

        n_survivors = len(candidates)

        while len(candidates) < max_candidates:
            candidate_mapping = np.copy(candidates[random.randint(0, n_survivors - 1)][1])
            random_swap(candidate_mapping)

            metric = compute_total_errors(candidate_mapping, confusion_matrix, hamming_matrix)
            candidates.append((metric, candidate_mapping))

        candidates.sort(key=key_function)
        candidates = candidates[:max_survivors]

    return candidates[0][1]

In [46]:
mapping = np.arange(0, 64, step=1, dtype=np.int8)
mapping = optimize_random_search(100_000_000, confusion_matrix, mapping)

In [47]:
compute_total_errors(np.arange(0, 64, step=1, dtype=np.int8), confusion_matrix, make_hamming_matrix(M)), compute_total_errors(mapping, confusion_matrix, make_hamming_matrix(M))

(7407070, 6037350)

In [11]:
mapping

array([59,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58,  0, 60, 61, 62, 63], dtype=int8)

In [44]:
_43[0] / _43[1]

1.2074247033466883

In [45]:
_12

1.0330898033451876